# Train U-Net Segmentation
Chest X-ray — Lung Mask Segmentation

In [1]:
# Sanity check GPU — chay truoc khi train
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

CUDA available: True
Device count: 1
Device name: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.546484224 GB
PyTorch: 2.8.0+cu129 CUDA: 12.9


In [2]:
# Set seed — dam bao reproducibility
import random, os
import numpy as np
import torch

def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
from pathlib import Path

from src.dataset import (
    ChestXraySegmentationDataset,
    get_train_transforms_seg, get_val_transforms_seg,
)
from src.unet import (
    build_unet, BCEDiceLoss, dice_score, iou_score,
)


In [ ]:
# CONFIG
SPLIT_DIR = "data/split"
BATCH_SIZE = 16   # U-Net tốn VRAM hơn classifier vì có thêm decoder
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT_PATH = "weights/best_unet.pth"
Path("weights").mkdir(exist_ok=True)

LR = 1e-4
EPOCHS = 25
PATIENCE = 5


In [ ]:
# DataLoader
train_ds = ChestXraySegmentationDataset(f"{SPLIT_DIR}/train", get_train_transforms_seg())
val_ds = ChestXraySegmentationDataset(f"{SPLIT_DIR}/val", get_val_transforms_seg())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
model = build_unet(pretrained=True).to(DEVICE)
criterion = BCEDiceLoss()
scaler = GradScaler()

def run_epoch(loader, train: bool, optimizer=None):
    model.train() if train else model.eval()
    losses, dices, ious = [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            losses.append(loss.item())
            dices.append(dice_score(logits, y))
            ious.append(iou_score(logits, y))
    return np.mean(losses), np.mean(dices), np.mean(ious)


In [ ]:
history = {
    "train_loss": [], "val_loss": [],
    "train_dice": [], "val_dice": [],
    "train_iou": [], "val_iou": [],
}


In [ ]:
# U-Net KHÔNG cần chia 3 pha như classifier: encoder ResNet-34 đã pretrained,
# decoder luôn train from scratch nên không có nguy cơ catastrophic forgetting
# (docs/TUTORIAL.md Phần 9, docs/QUY_TRINH_CODE.md Phần 5.4) — unfreeze toàn bộ ngay từ đầu.
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_dice = 0.0
patience_ctr = 0

for ep in range(EPOCHS):
    tr_loss, tr_dice, tr_iou = run_epoch(train_loader, train=True, optimizer=optimizer)
    va_loss, va_dice, va_iou = run_epoch(val_loader, train=False)
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_dice"].append(tr_dice)
    history["val_dice"].append(va_dice)
    history["train_iou"].append(tr_iou)
    history["val_iou"].append(va_iou)

    scheduler.step()
    print(f"Ep {ep+1:02d} train_loss={tr_loss:.4f} tr_dice={tr_dice:.4f} tr_iou={tr_iou:.4f} "
          f"val_loss={va_loss:.4f} val_dice={va_dice:.4f} val_iou={va_iou:.4f}")

    if va_dice > best_dice:
        best_dice = va_dice
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"  Saved best_dice={best_dice:.4f}")
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  Early stop at epoch {ep+1}")
            break

print(f"\nBEST VAL DICE: {best_dice:.4f}")


In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. Loss, Dice, IoU theo epoch
# ============================================================
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, history["train_loss"], marker="o", label="Train loss")
axes[0].plot(epochs, history["val_loss"], marker="o", label="Val loss")
axes[0].set_title("Loss theo epoch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE + Dice loss")
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(epochs, history["train_dice"], marker="o", label="Train Dice")
axes[1].plot(epochs, history["val_dice"], marker="o", label="Val Dice")
axes[1].set_title("Dice theo epoch")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice"); axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3); axes[1].legend()

axes[2].plot(epochs, history["train_iou"], marker="o", label="Train IoU")
axes[2].plot(epochs, history["val_iou"], marker="o", label="Val IoU")
axes[2].set_title("IoU theo epoch")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("IoU"); axes[2].set_ylim(0, 1)
axes[2].grid(alpha=0.3); axes[2].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "unet_loss_dice_iou_curves.png", dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 2. Sanity check định tính: 5 ảnh mẫu [gốc | mask thật | mask dự đoán]
#    (docs/TUTORIAL.md Phần 9.4 — bắt buộc trước khi coi U-Net "pass")
# ============================================================
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

n_samples = 5
fig, axes = plt.subplots(n_samples, 3, figsize=(9, 3 * n_samples))

with torch.no_grad():
    for i in range(n_samples):
        img, mask = val_ds[i]
        logits = model(img.unsqueeze(0).to(DEVICE))
        pred = (torch.sigmoid(logits)[0, 0] > 0.5).cpu().numpy()

        img_show = img.permute(1, 2, 0).numpy()
        img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min() + 1e-8)

        axes[i, 0].imshow(img_show); axes[i, 0].set_title("Ảnh gốc"); axes[i, 0].axis("off")
        axes[i, 1].imshow(mask[0].numpy(), cmap="gray"); axes[i, 1].set_title("Mask thật"); axes[i, 1].axis("off")
        axes[i, 2].imshow(pred, cmap="gray"); axes[i, 2].set_title("Mask dự đoán"); axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "unet_qualitative_check.png", dpi=300, bbox_inches="tight")
plt.show()
